In [72]:
# !pip install geopy

In [73]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Duomenų paruošimas

### Stočių ir matavimų duomenys

In [74]:
station = pd.read_csv("Station.csv")
station = station.drop(["_type", "_revision", "_page.next"], axis=1)
station.head()

,_id,stat_num,latitude,longitude
0,cb2cef5d-4776-4341-941d-07ad0a398b0b,1,54.677625,25.285186
1,6bd59444-5437-447d-bc2d-c32ef5b0a964,2,54.686111,25.210835
2,ed74723a-e2dd-41c4-a3b0-ae7c2cb0695c,3,54.715278,25.289444
3,53f89320-7ac7-4c90-9612-95b663284fb7,4,54.673333,25.248903
4,d3a305f2-183b-4d20-8b28-f2e71f885ad6,6,NaN,NaN


Iš koordinačių ištraukiami stočių miestai.

In [75]:
geolocator = Nominatim(user_agent="geo_project")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_city(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan

    location = reverse((lat, lon), language="en")

    if location is None:
        return np.nan

    address = location.raw['address']

    return (
        address.get('city') or
        address.get('town') or
        address.get('village') or
        address.get('municipality') or
        address.get('county') or
        np.nan
    )

station['city'] = station.apply(lambda x: get_city(x['latitude'], x['longitude']), axis=1)

In [76]:
station

,_id,stat_num,latitude,longitude,city
0,cb2cef5d-4776-4341-941d-07ad0a398b0b,1,54.677625,25.285186,Vilnius
1,6bd59444-5437-447d-bc2d-c32ef5b0a964,2,54.686111,25.210835,Vilnius
2,ed74723a-e2dd-41c4-a3b0-ae7c2cb0695c,3,54.715278,25.289444,Vilnius
3,53f89320-7ac7-4c90-9612-95b663284fb7,4,54.673333,25.248903,Vilnius
4,d3a305f2-183b-4d20-8b28-f2e71f885ad6,6,NaN,NaN,NaN
5,27cbf611-ff8f-4851-bf66-e87d96303ad8,11,NaN,NaN,NaN
6,8628cd14-cc2c-4b7a-a17d-98e9b9d42083,12,55.725000,24.365569,Panevėžys
7,f39d3d59-6a18-4d34-954a-d39e4c8f86a5,21,56.319440,22.870840,Naujoji Akmenė
8,37d8b588-c538-4db7-be57-dfd1527c6588,22,55.937833,23.308028,Šiauliai
9,c533d7e0-d443-48e6-b123-6da13ca87db6,23,56.309722,22.331389,Mažeikiai


In [77]:
averages = pd.read_csv("Averages.csv")
averages = averages.drop(["_type", "_revision", "_page.next"], axis=1)
averages.head()

C:\Users\crist\AppData\Local\Temp\ipykernel_37040\391105750.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  averages = pd.read_csv("Averages.csv")


,_id,id,stat_num._id,ldatetime,code_combi,lvalue,atribut
0,0883565b-c0b7-490d-8687-f73ed09a5bc6,100000004,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,6201,0.00,1
1,78ed962e-9b5e-4b2e-91e1-af211488a26b,100000005,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,6901,0.00,1
2,dbb449c9-84c9-40e2-b7e8-56fe24d7043d,100000009,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,4501,1.03,1
3,5e33d873-cbd7-4715-b8e5-0769f65dd321,100000010,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,7101,0.01,1
4,703a188a-f59b-496c-accf-75c38b13c74c,100000011,6bd59444-5437-447d-bc2d-c32ef5b0a964,2012-12-11T10:00:00,5201,0.30,1


Stotys apjungiamos su pagrindine matavymų lentele.

In [ ]:
averages_stations = averages.merge(
    station[['_id', 'stat_num', 'city']], 
    left_on='stat_num._id', 
    right_on='_id', 
    how='left'
)

averages_stations = averages_stations.drop(["stat_num._id", "_id_y"], axis=1)
averages_stations.head()

,_id_x,id,ldatetime,code_combi,lvalue,atribut,stat_num,city
0,0883565b-c0b7-490d-8687-f73ed09a5bc6,100000004,2012-12-11T10:00:00,6201,0.00,1,2,Vilnius
1,78ed962e-9b5e-4b2e-91e1-af211488a26b,100000005,2012-12-11T10:00:00,6901,0.00,1,2,Vilnius
2,dbb449c9-84c9-40e2-b7e8-56fe24d7043d,100000009,2012-12-11T10:00:00,4501,1.03,1,2,Vilnius
3,5e33d873-cbd7-4715-b8e5-0769f65dd321,100000010,2012-12-11T10:00:00,7101,0.01,1,2,Vilnius
4,703a188a-f59b-496c-accf-75c38b13c74c,100000011,2012-12-11T10:00:00,5201,0.30,1,2,Vilnius


In [79]:
missing_cities = averages_stations['city'].isna().sum()
print(f"Eilutės be miesto: {missing_cities}")

if missing_cities > 0:
    print(averages_stations[averages_stations['city'].isna()]['stat_num'].unique())


Eilutės be miesto: 37660
[99 11 32  6]


### Taršos ir matavimo vienetų duomenys

In [80]:
units= pd.read_csv("Units.csv")
units = units.drop(["_type", "_revision", "_page.next"], axis=1)
units

,_id,code_unit,unitname,id
0,28f650e6-08d2-4311-98fe-55cc2aba9ff0,1,ppb,1
1,48518a60-6e96-4b6f-93b8-bcaeff67685a,2,ppm,2
2,89a0ce55-265d-4266-91ea-5eb0ed25b77b,3,ug/m3,3
3,0fc540c4-cae8-4557-a091-6e310dd8d095,4,m/s,4
4,85e37543-6901-475a-83e1-0647bbd4b2bc,5,deg,5
5,67461e7c-506e-430a-8962-cd25ebed54da,6,hPa,6
6,93b8a619-9e3b-4204-8957-4dee6a075d4a,7,^C,7
7,866e2570-2343-40fa-b0e7-999c75f9a46e,8,%,8
8,4797ee11-9fda-4178-930f-13ad3ae435a6,9,W/m2,9
9,2b1b3b00-a4d3-4ea4-9dff-92374f6fc07b,10,kod,10


In [81]:
quantity = pd.read_csv("Quantity.csv")
quantity = quantity.drop(["_type", "_revision", "_page.next"], axis=1)
quantity.head()

,_id,code_quantity,shortname,longname,attrib,code_unit._id,id
0,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,1,SO2,sulphur dioxide,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,1
1,686bb9d9-e770-4034-b6d9-f75a6d922141,2,NO,nitrogen monoxide,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,2
2,ad7c3c3e-8cd7-4f15-b89b-669e28660296,3,NO2,nitrogen dioxide,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,3
3,f04d89f4-440d-4b6b-b7c2-0331245f6940,4,NOx,nitrogen oxides,gasep,28f650e6-08d2-4311-98fe-55cc2aba9ff0,4
4,1fefab1d-1525-41a8-8a03-d82b041f4dbc,5,PM10,PM10 particulates,parti,89a0ce55-265d-4266-91ea-5eb0ed25b77b,5


In [82]:
quantity_units = pd.read_csv("QuantityUnits.csv")
quantity_units = quantity_units.drop(["_type", "_revision", "_page.next"], axis=1)
quantity_units.head()

,_id,code_combi,code_quantity._id,code_unit._id
0,dcc66f7a-4802-4f7d-b9bc-f790137c8613,101,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,28f650e6-08d2-4311-98fe-55cc2aba9ff0
1,02643e24-3205-4af1-a608-76009dacff59,102,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,48518a60-6e96-4b6f-93b8-bcaeff67685a
2,d55fe2d8-d1dc-44aa-8433-b520480bd5d7,103,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,89a0ce55-265d-4266-91ea-5eb0ed25b77b
3,24f92792-dc63-4b2c-a715-69410ebd1f02,104,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,0fc540c4-cae8-4557-a091-6e310dd8d095
4,f6e05409-ede2-45a9-b5d8-5d260f28af48,105,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,85e37543-6901-475a-83e1-0647bbd4b2bc


In [83]:
full_units = pd.merge(
    quantity_units, 
    units[['_id', 'code_unit', 'unitname']], 
    left_on='code_unit._id', 
    right_on='_id', 
    how='left'
)

full_units.head()

,_id_x,code_combi,code_quantity._id,code_unit._id,_id_y,code_unit,unitname
0,dcc66f7a-4802-4f7d-b9bc-f790137c8613,101,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,28f650e6-08d2-4311-98fe-55cc2aba9ff0,28f650e6-08d2-4311-98fe-55cc2aba9ff0,1,ppb
1,02643e24-3205-4af1-a608-76009dacff59,102,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,48518a60-6e96-4b6f-93b8-bcaeff67685a,48518a60-6e96-4b6f-93b8-bcaeff67685a,2,ppm
2,d55fe2d8-d1dc-44aa-8433-b520480bd5d7,103,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,89a0ce55-265d-4266-91ea-5eb0ed25b77b,89a0ce55-265d-4266-91ea-5eb0ed25b77b,3,ug/m3
3,24f92792-dc63-4b2c-a715-69410ebd1f02,104,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,0fc540c4-cae8-4557-a091-6e310dd8d095,0fc540c4-cae8-4557-a091-6e310dd8d095,4,m/s
4,f6e05409-ede2-45a9-b5d8-5d260f28af48,105,0b30092e-0bac-4a8a-a21b-ea3916ae6e85,85e37543-6901-475a-83e1-0647bbd4b2bc,85e37543-6901-475a-83e1-0647bbd4b2bc,5,deg


In [84]:
full_quantity_units= pd.merge(
    full_units, 
    quantity[['_id', 'code_quantity', 'shortname', 'longname', 'attrib']], 
    left_on='code_quantity._id', 
    right_on='_id', 
    how='left',
    suffixes=('', '_qty')
)

full_quantity_units = full_quantity_units.drop(["_id_x", "code_quantity._id", "code_unit._id", "_id_y", "_id"], axis=1)
full_quantity_units.head()

,code_combi,code_unit,unitname,code_quantity,shortname,longname,attrib
0,101,1,ppb,1,SO2,sulphur dioxide,gasep
1,102,2,ppm,1,SO2,sulphur dioxide,gasep
2,103,3,ug/m3,1,SO2,sulphur dioxide,gasep
3,104,4,m/s,1,SO2,sulphur dioxide,gasep
4,105,5,deg,1,SO2,sulphur dioxide,gasep


### Jungimas į vieną bendrą rinkinį

In [85]:
air_quality_data = pd.merge(
    averages_stations, 
    full_quantity_units, 
    left_on='code_combi', 
    right_on='code_combi', 
    how='left'
)
air_quality_data = air_quality_data.drop(["_id_x", "id", "code_combi", "code_unit", "code_quantity", "atribut"], axis=1)
air_quality_data.head()

,ldatetime,lvalue,stat_num,city,unitname,shortname,longname,attrib
0,2012-12-11T10:00:00,0.00,2,Vilnius,ppb,ISBTEN,iso-butene-PID,Oprek
1,2012-12-11T10:00:00,0.00,2,Vilnius,ppb,ISPREN,isoprene-PID,Oprek
2,2012-12-11T10:00:00,1.03,2,Vilnius,ppb,NHPTAN,n-heptane,Oprek
3,2012-12-11T10:00:00,0.01,2,Vilnius,ppb,NHXA1,n-hexane,Oprek
4,2012-12-11T10:00:00,0.30,2,Vilnius,ppb,NHXAN,n-hexane,Oprek


In [86]:
air_quality_data['ldatetime'] = pd.to_datetime(air_quality_data['ldatetime'])
air_quality_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2643089 entries, 0 to 2643088
Data columns (total 8 columns):
 #   Column     Dtype         
---  ------     -----         
 0   ldatetime  datetime64[ns]
 1   lvalue     float64       
 2   stat_num   int64         
 3   city       object        
 4   unitname   object        
 5   shortname  object        
 6   longname   object        
 7   attrib     object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 161.3+ MB


# Praleistos reikšmės

In [87]:
station_summary = air_quality_data.groupby(['stat_num']).agg(
    pradzia=('ldatetime', 'min'),
    pabaiga=('ldatetime', 'max'),
    matavimu_skaicius=('lvalue', 'count')
).reset_index()

print(station_summary)

    stat_num             pradzia             pabaiga  matavimu_skaicius
0          1 2002-02-02 17:30:00 2023-11-10 18:00:00              34362
1          2 2002-01-28 02:00:00 2023-11-09 21:00:00             972863
2          3 2002-02-09 13:00:00 2023-11-09 21:00:00             126477
3          4 2002-01-27 01:30:00 2023-11-10 15:00:00              94944
4          6 2002-05-02 13:30:00 2006-12-12 15:00:00               1489
5         11 2002-01-27 00:30:00 2008-08-21 06:30:00              21672
6         12 2005-07-29 01:00:00 2023-11-09 21:00:00              65515
7         21 2002-03-15 13:00:00 2023-11-10 19:00:00              79779
8         22 2002-03-21 13:00:00 2023-11-10 12:00:00              57767
9         23 2002-09-05 19:30:00 2023-11-10 09:00:00              26050
10        31 2002-01-30 12:30:00 2023-11-10 18:00:00              79368
11        32 2002-02-08 23:30:00 2005-06-22 14:00:00                756
12        33 2005-06-30 13:30:00 2023-11-10 12:00:00            

In [88]:
air_quality_data["shortname"].unique()

array(['ISBTEN', 'ISPREN', 'NHPTAN', 'NHXA1', 'NHXAN', 'PRPEN', 'T2PTEN',
       'TLN', '224TMP', '2MPTA1', '2MPTAN', 'ETHAN', 'ETHEN', 'ETHYN',
       'HUMI', 'PRES', 'TEMP', 'NO', 'NO2', 'NOx', 'O3', 'WD', 'WV',
       'PM10', 'Tint', 'BZN', 'MPXY', 'OXY', 'PM25', 'GLRD', 'C2BTEN',
       'IBTAN', 'IPTAN', 'NBTAN', 'NOCTAN', 'NPTAN', 'T2BTEN', 'EBZN',
       'HgA', 'HgB', '123TMB', '124TMB', '135TMB', '13BTDN', '1BTEN',
       '1PTEN', 'PRPAN', 'CO', 'SO2', nan, 'flow', 'Slegis1', 'Slegis2',
       'Slegis3', 'Slegis4', 'Nh01', 'Nh02', 'Nh05', 'CH2O', 'Ligh',
       'Tdet', 'CSpe', 'NH3', 'H2S', 'Cn', 'PM1', 'PM4', 'PM_total'],
      dtype=object)

In [89]:
air_quality_data.head()


,ldatetime,lvalue,stat_num,city,unitname,shortname,longname,attrib
0,2012-12-11 10:00:00,0.00,2,Vilnius,ppb,ISBTEN,iso-butene-PID,Oprek
1,2012-12-11 10:00:00,0.00,2,Vilnius,ppb,ISPREN,isoprene-PID,Oprek
2,2012-12-11 10:00:00,1.03,2,Vilnius,ppb,NHPTAN,n-heptane,Oprek
3,2012-12-11 10:00:00,0.01,2,Vilnius,ppb,NHXA1,n-hexane,Oprek
4,2012-12-11 10:00:00,0.30,2,Vilnius,ppb,NHXAN,n-hexane,Oprek


In [90]:
air_quality_data.groupby(['shortname']).agg(
    num_nas=('lvalue', lambda x: x.isna().sum()),
    total_count=('lvalue', 'count')
).sort_values(by='num_nas', ascending=False).reset_index()

,shortname,num_nas,total_count
0,123TMB,0,23296
1,124TMB,0,54925
2,135TMB,0,65972
3,13BTDN,0,11314
4,1BTEN,0,14822
...,...,...,...
62,Tdet,0,872
63,Tint,0,32526
64,WD,0,194953
65,WV,0,197255


In [91]:
# air_quality_data.to_csv("air_quality_data.csv", index=False)

In [92]:
air_quality_data_Vilnius = air_quality_data[air_quality_data['city'] == 'Vilnius']
air_quality_data_Vilnius.head()


,ldatetime,lvalue,stat_num,city,unitname,shortname,longname,attrib
0,2012-12-11 10:00:00,0.00,2,Vilnius,ppb,ISBTEN,iso-butene-PID,Oprek
1,2012-12-11 10:00:00,0.00,2,Vilnius,ppb,ISPREN,isoprene-PID,Oprek
2,2012-12-11 10:00:00,1.03,2,Vilnius,ppb,NHPTAN,n-heptane,Oprek
3,2012-12-11 10:00:00,0.01,2,Vilnius,ppb,NHXA1,n-hexane,Oprek
4,2012-12-11 10:00:00,0.30,2,Vilnius,ppb,NHXAN,n-hexane,Oprek


In [93]:
air_quality_data_Vilnius = pd.pivot_table(
    air_quality_data_Vilnius,
    index='ldatetime',
    columns='shortname',
    values='lvalue'
).reset_index()

air_quality_data_Vilnius.head()

shortname,ldatetime,123TMB,124TMB,135TMB,13BTDN,1BTEN,1PTEN,224TMP,2MPTA1,2MPTAN,...,Slegis4,T2BTEN,T2PTEN,TEMP,TLN,Tdet,Tint,WD,WV,flow
0,2002-01-27 01:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2002-01-27 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002-01-27 03:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2002-01-27 08:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2002-01-27 13:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
air_quality_data_Vilnius.iloc[:, 1:].isna().sum().to_frame(name='num_nas').reset_index().rename(columns={'index': 'shortname'}).sort_values(by='num_nas', ascending=False).reset_index(drop=True)

,shortname,num_nas
0,Tdet,119709
1,Ligh,119708
2,CH2O,119687
3,Cn,119669
4,PM4,119669
...,...,...
57,224TMP,59183
58,NOCTAN,58813
59,2MPTAN,57997
60,TLN,55347


In [95]:
air_quality_data_Vilnius.assign(year=air_quality_data_Vilnius['ldatetime'].dt.year) \
    .groupby('year') \
    .apply(lambda x: pd.Series({
        'num_nas': x.iloc[:, 1:].isna().sum().sum(),
        'pct_nas': x.iloc[:, 1:].isna().sum().sum() / x.iloc[:, 1:].size * 100
    })) \
    .reset_index()

C:\Users\crist\AppData\Local\Temp\ipykernel_37040\395522846.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,year,num_nas,pct_nas
0,2002,255913.0,94.643782
1,2003,55453.0,96.197415
2,2004,15175.0,93.724909
3,2005,13530.0,93.782491
4,2006,49783.0,95.898829
5,2007,153199.0,94.216589
6,2008,442462.0,80.292744
7,2009,397784.0,81.555564
8,2010,518300.0,93.947518
9,2011,159362.0,89.163044


In [96]:
# air_quality_data_Vilnius.to_csv("air_quality_data_Vilnius.csv", index=False)

In [97]:
for stat, group in air_quality_data[air_quality_data['city'] == 'Vilnius'].groupby('stat_num'):
    na_summary = (
        group.pivot_table(index='ldatetime', columns='shortname', values='lvalue')
        .isna()
        .mean() * 100
    )
    print(f"\n--- Station {stat} ---")
    print(na_summary.sort_values().head(10).to_frame(name='pct_nas').round(2))


--- Station 1 ---
           pct_nas
shortname         
TEMP         69.50
NO           75.90
NOx          76.00
NO2          77.70
PM10         77.92
Tint         78.20
HUMI         79.50
PRES         81.19
WV           82.79
WD           83.14

--- Station 2 ---
           pct_nas
shortname         
135TMB       32.32
2MPTAN       36.68
TLN          37.47
NOCTAN       37.51
224TMP       37.89
124TMB       43.65
NHPTAN       47.56
EBZN         48.33
MPXY         48.53
OXY          50.14

--- Station 3 ---
           pct_nas
shortname         
Nh05         63.22
Nh02         63.57
BZN          69.88
Nh01         78.41
PRES         79.68
TEMP         82.12
HUMI         82.40
WV           82.77
WD           82.82
TLN          89.01

--- Station 4 ---
           pct_nas
shortname         
PRES         37.95
TEMP         37.95
HUMI         38.09
WD           39.05
WV           39.05
TLN          78.93
OXY          82.75
MPXY         82.87
PM10         88.12
NO           91.53


In [98]:
particles_of_interest = ['PM_total', 'PM4', 'PM1', 'PM25', 'DUST', 'PM10', 'CN', 
                         'SO2', 'NOx', 'NH3', 'THC', 'NMHC', 'VOC', 'TRS', 'HgA', 'HgB']

for stat, group in air_quality_data[air_quality_data['city'] == 'Vilnius'].groupby('stat_num'):
    na_summary = (
        group.pivot_table(index='ldatetime', columns='shortname', values='lvalue')
        .reindex(columns=particles_of_interest)
        .isna()
        .mean() * 100
    )
    print(f"\n--- Station {stat} ---")
    print(na_summary.sort_values().to_frame(name='pct_nas').round(2))


--- Station 1 ---
           pct_nas
shortname         
NOx          76.00
PM10         77.92
SO2          88.11
PM_total    100.00
PM25        100.00
PM1         100.00
DUST        100.00
PM4         100.00
CN          100.00
NH3         100.00
THC         100.00
NMHC        100.00
VOC         100.00
TRS         100.00
HgA         100.00
HgB         100.00

--- Station 2 ---
           pct_nas
shortname         
NOx          98.26
PM10         98.44
SO2          99.22
PM_total    100.00
PM25        100.00
PM1         100.00
DUST        100.00
PM4         100.00
CN          100.00
NH3         100.00
THC         100.00
NMHC        100.00
VOC         100.00
TRS         100.00
HgA         100.00
HgB         100.00

--- Station 3 ---
           pct_nas
shortname         
PM25         94.32
NOx          96.22
PM10         97.45
SO2          99.67
PM1          99.89
PM4          99.89
PM_total     99.89
DUST        100.00
CN          100.00
NH3         100.00
THC         100.00
NMHC        

In [67]:
# Find unmatched code_combis
unmatched = averages_stations[~averages_stations['code_combi'].isin(full_quantity_units['code_combi'])]

print(unmatched['code_combi'].value_counts())
print(unmatched['city'].value_counts())  # which cities are affected?
print(unmatched['stat_num'].value_counts())  # which stations?

code_combi
7420     34955
8620     20459
7903     18816
7808      9485
8506      1142
7701       688
7501       688
7601       688
7801       666
99999      437
8601       378
7726       172
7526       172
7626       172
7703       162
7603       162
7503       162
8503        97
7403        51
7901        49
8303        46
8103        46
8003        46
8203        46
8501        36
8603        32
8403        20
8102         4
8202         4
8302         4
8402         4
8002         4
7803         2
8602         2
7826         2
Name: count, dtype: int64
city
Vilnius           36893
Naujoji Akmenė    14531
Klaipėda          12870
Plokščiai         11930
Panevėžys          5022
Kėdainiai          2138
Šiauliai           1420
Mažeikiai           759
Noreikiškės         756
Kaunas              727
Jonava              653
Rūgšteliškis        160
Dubininkas           74
Name: count, dtype: int64
stat_num
4     21123
21    14531
53    11930
31    10788
3      8413
12     5022
2      4045
1 

In [68]:
unmatched[unmatched['city'] == 'Vilnius']['stat_num'].value_counts()

stat_num
4    21123
3     8413
2     4045
1     3312
Name: count, dtype: int64